## Import libraries

In [ ]:
import pandas as pd
import numpy as np

## Paths

In [ ]:
data_path = "../data/processed"

In [ ]:
from eda_toolkit import ensure_directory
import os  # import operating system for dir

base_path = os.path.join(os.pardir)

# Go up one level from 'notebooks' to parent directory,
# then into the 'data' folder
data_path = os.path.join(os.pardir, "data/processed")

# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_pdf = os.path.join(base_path, "images", "pdf_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")

# Use the function to ensure'data' directory exists
ensure_directory(data_path)
ensure_directory(image_path_png)
ensure_directory(image_path_pdf)
ensure_directory(image_path_svg)

In [ ]:
import json
from pathlib import Path

pred_dir = Path("../models/predictions/full_text_clean")

X = pd.read_parquet("../data/processed/X.parquet")
y = pd.read_parquet("../data/processed/y.parquet").squeeze()

In [ ]:
from model_metrics.model_registry import set_stores

set_stores("mlruns/models")                     # only the live store

In [ ]:
from model_metrics.model_registry import best_per_algo, load_best_per_algo

best_per_algo(metric="valid Average Precision")
champs = load_best_per_algo(metric="valid Average Precision")

In [ ]:
champs

In [ ]:
model_catboost = champs["cat_outcome"]
model_catboost_no_sex = champs["cat_outcome_no_sex"]
model_xgboost = champs["xgb_outcome"]
model_rf = champs["rf_outcome"]
model_lr = champs["lr_outcome"]

In [ ]:
model_titles = ["CatBoost", "CatBoost (sex removed)", "XGBoost", "Random Forest", "Logistic Regression"]
models = [model_catboost, model_catboost_no_sex, model_xgboost, model_rf, model_lr,]

In [ ]:
thresholds = {
    "Logistic Regression": next(iter(model_lr.threshold.values())),
    "Random Forest": next(iter(model_rf.threshold.values())),
    "XGBoost": next(iter(model_xgboost.threshold.values())),
    "CatBoost": next(iter(model_catboost.threshold.values())),
    "CatBoost (sex removed)": next(iter(model_catboost_no_sex.threshold.values())),
}

In [ ]:
X_valid, y_valid = model_lr.get_valid_data(X, y)
X_test, y_test = model_lr.get_test_data(X, y)
y_test = y_test["outcome"]

In [ ]:
from model_metrics import get_expected_features

get_expected_features(model_rf)

In [ ]:
"""
SHAP feature attributions for the primary CatBoost model.

The model is wrapped by model_tuner and calibrated, so `predict_proba` does
not expose a bare CatBoost booster. TreeExplainer needs the booster itself and
features in the space the booster saw, which is post-preprocessing. Both are
extracted below rather than assumed.

Calibration sits downstream of the booster, so these values explain the
uncalibrated log-odds rather than the calibrated probability. That is standard
and worth one sentence in the Methods.
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from pathlib import Path

MODEL = model_catboost
OUT_PNG = "../images/png_images"
OUT_PDF = "../images/pdf_images"

# --- unwrap: model_tuner -> calibrated wrapper -> Pipeline -> booster -------
est = getattr(MODEL, "estimator", MODEL)

if hasattr(est, "calibrated_classifiers_"):
    inner = est.calibrated_classifiers_[0]
    est = getattr(inner, "estimator", inner)

pipeline = est
booster = pipeline.steps[-1][1]
preprocessor = pipeline[:-1]

print("booster:", type(booster).__name__)
print("preprocessing steps:", [name for name, _ in pipeline.steps[:-1]])

# --- transform into the booster's feature space ----------------------------
X_trans = preprocessor.transform(X_test)

try:
    feature_names = list(preprocessor.get_feature_names_out())
except Exception:
    feature_names = list(X_test.columns)

# strip the ColumnTransformer prefix so labels read as clinical variables
feature_names = [n.split("__")[-1] for n in feature_names]

X_shap = pd.DataFrame(np.asarray(X_trans), columns=feature_names,
                      index=X_test.index)

PRETTY = {
    "creatinine": "Serum creatinine",
    "cardiovascular_disease": "Cardiovascular disease",
    "diabetes": "Diabetes mellitus",
    "dyslipidemia": "Dyslipidemia",
    "smoking": "Smoking",
    "cancer": "Cancer",
    "hypertension": "Hypertension",
    "sex": "Sex",
    "obesity": "Obesity",
}
X_shap = X_shap.rename(columns=PRETTY)
feature_names = list(X_shap.columns)

print(f"explaining {X_shap.shape[0]} patients on {X_shap.shape[1]} features")

# --- SHAP values -----------------------------------------------------------
explainer = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(X_shap)

# binary CatBoost returns one array; some versions return a list of two
if isinstance(shap_values, list):
    shap_values = shap_values[1]

print("shap values shape:", np.shape(shap_values))

Path(OUT_PNG).mkdir(parents=True, exist_ok=True)
Path(OUT_PDF).mkdir(parents=True, exist_ok=True)

# --- beeswarm: direction and magnitude per patient -------------------------
plt.figure(figsize=(7, 5))
shap.summary_plot(shap_values, X_shap, plot_type="dot", show=False,
                  max_display=len(feature_names))
plt.title("SHAP values, primary model", fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUT_PNG}/figure_shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUT_PDF}/figure_shap_beeswarm.pdf", bbox_inches="tight")
plt.show()

# --- bar: mean absolute contribution ---------------------------------------
plt.figure(figsize=(7, 4.5))
shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False,
                  max_display=len(feature_names))
plt.title("Mean absolute SHAP value, primary model", fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUT_PNG}/figure_shap_bar.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUT_PDF}/figure_shap_bar.pdf", bbox_inches="tight")
plt.show()

# --- ranking table, so the Results text quotes numbers not a picture -------
importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Mean |SHAP|": np.abs(shap_values).mean(axis=0),
        "Mean SHAP": np.asarray(shap_values).mean(axis=0),
    })
    .sort_values("Mean |SHAP|", ascending=False)
    .reset_index(drop=True)
)
importance["Rank"] = importance.index + 1
print()
print(importance.round(4).to_string(index=False))

# --- attribution by sex, which speaks to the fairness analysis -------------
by_sex = pd.DataFrame(np.abs(shap_values), columns=feature_names,
                      index=X_test.index)
sex_labels = X_test["sex"].map({0: "Female", 1: "Male"}).values

sex_importance = (
    by_sex.groupby(sex_labels).mean().T
    .assign(Difference=lambda d: d["Female"] - d["Male"])
    .sort_values("Female", ascending=False)
)
sex_importance.index.name = "Feature"
print()
print("Mean |SHAP| by sex:")
print(sex_importance.round(4).to_string())